# quality

> feedback, a noise score, and a ranker fitted from both

In [ ]:
#| default_exp quality

Three things live here, none of them on by default.

```python
v.learn()                  # log every ask as feedback
v.suggest_noisy(k=20)      # documents that look like junk
v.fit_ranker(save=True)    # fit a ranker over the log
v.use_ranker(True)         # put it in the path of your questions
```

`mark_noisy` already covers the case where you have seen a document and rejected it. This module
covers the two you cannot do by hand: finding the junk you have not read yet, and noticing the
ranking is wrong.

What ships switched on is decided by `evals/`. Right now that is the noise score and nothing
else. Numbers in `evals/RESULTS.md`.

In [ ]:
#| export
import json, math, re, time, uuid, warnings
import numpy as np
from collections import Counter, defaultdict
from fastcore.all import AttrDict, L, patch, first
from vishalakshi.core import Vault, DTYPE

## What was good

`SIGNALS` maps a signal to a label and a weight. A thumb is worth 3, a citation 1, and "shown and
not used" 0.35.

That last one is the weak negative, and it is the one people leave out. A log of positives
teaches a ranker that everything is relevant, because it has never seen anything that was not.

In [ ]:
#| export
#: signal -> (label, weight). `shown` is the weak negative every implicit log needs and most
#: implicit logs omit. The weights are a starting prior, not a measurement: `evals/run.py`
#: refits with them varied, and they are cheap to change because nothing downstream caches them.
SIGNALS = dict(up=(1.0, 3.0), good=(1.0, 3.0), read=(1.0, 1.5), cited=(1.0, 1.0),
               shown=(0.0, 0.35), bad=(0.0, 3.0), down=(0.0, 3.0))

@patch
def _fb(self:Vault):
    "The feedback log, created on first use: one row per (question, section) judgement."
    t = self.db.t.feedback
    t.create(id=str, at=float, store=str, q=str, ask_id=str, node_id=str, doc_id=str, rank=int,
             score=float, signal=str, label=float, weight=float, pk='id', if_not_exists=True)
    for cols in ('store, doc_id', 'store, ask_id'):
        try: self.db.conn.execute(f"CREATE INDEX IF NOT EXISTS feedback_{cols.split(', ')[1]} ON feedback({cols})")
        except Exception: pass
    return t

@patch
def rate(self:Vault,
         q:str,               # the question this judgement is about
         node_id:str=None,    # the section being judged
         doc_id:str=None,     # its document; derived from `node_id` when not given
         signal:str='up',     # one of SIGNALS
         rank:int=None,       # where it was ranked when it was shown
         score:float=None,    # the retrieval score it was shown with
         ask_id:str=None,     # groups the rows from one question together
) -> dict:
    "Record one judgement. `v.rate(q, node_id, signal='down')` is the whole manual interface."
    if signal not in SIGNALS: raise ValueError(f'not a signal: {signal!r}; expected {sorted(SIGNALS)}')
    label, weight = SIGNALS[signal]
    row = dict(id=uuid.uuid4().hex[:16], at=time.time(), store=self.name, q=q or '',
               ask_id=ask_id or uuid.uuid4().hex[:12], node_id=node_id,
               doc_id=doc_id or (node_id.split('#')[0] if node_id else None),
               rank=rank, score=score, signal=signal, label=label, weight=weight)
    self._fb().insert(row)
    return row

@patch
def ratings(self:Vault, doc_id:str=None, limit:int=None) -> L:
    "The feedback log for this shelf, newest first."
    w = f'store={self.name!r}' + (f' AND doc_id={doc_id!r}' if doc_id else '')
    return L(self._fb()(where=w, order_by='at desc', limit=limit))

### Citations are the label

`ask` already returns `cited`: which sections the answer used. That is a relevance judgement, on
the right question, produced free on every ask. `log_ask` writes it down.

Two rules, both of which cost something to get wrong.

An answer that cites nothing is not logged at all. Small models drop the citation convention when
they are unsure, so logging that as six bad sections fills the log with the retrievals for the
hardest questions.

Citations are position-biased: the model reads `[1]` first and cites it more. So this log is
training data and never a scoreboard. `evals/` scores on a separate gold set.

In [ ]:
#| export
@patch
def log_ask(self:Vault,
            out,             # what `ask` returned
            up:list=None,    # section numbers you thought were good, as `[1]`-style numbers
            down:list=None,  # section numbers you thought were bad
            weak:bool=True,  # log the uncited sections as weak negatives
) -> int:
    """Turn one answered question into labels. Returns how many rows were written.

    The numbers in `up`/`down` are the ones in the answer and in `mk_prompt`: `[1]` is the first
    section: so a person can say `v.log_ask(out, down=[2])` while reading, without looking
    anything up."""
    up, down = set(up or ()), set(down or ())
    res = L((out.get('context') or {}).get('results') or ())
    cited = {c['n'] for c in (out.get('cited') or L())}
    if not (cited or up or down): return 0     # see above: no citations is not six negatives
    q, aid = out.get('question') or '', uuid.uuid4().hex[:12]
    n = 0
    for i, r in enumerate(res, 1):
        sig = ('up' if i in up else 'down' if i in down else 'cited' if i in cited else 'shown' if weak else None)
        if sig is None or not getattr(r, 'node_id', None): continue
        self.rate(q, node_id=r.node_id, doc_id=getattr(r, 'doc_id', None), signal=sig, rank=i,
                  score=float(getattr(r, 'score', 0.0) or 0.0), ask_id=aid)
        n += 1
    return n

@patch
def _observe(self:Vault, out):
    "The hook `ask` calls on its way out; silent unless `v.learn()` turned logging on."
    if getattr(self, '_learning', None) is None:
        try: self._learning = bool((first(self._rankers()(where=f'store={self.name!r}')) or {}).get('logging'))
        except Exception: self._learning = False
    if self._learning:
        try: self.log_ask(out)
        except Exception as e: warnings.warn(f'feedback not logged ({type(e).__name__}: {e})')
    return out

## The Beta prior

`Beta(1+wins, 1+losses)` per document, turned into a score multiplier by `(p/0.5)**gamma`.
Counting, before any model.

It helps in one regime and hurts in the other. Where the same material is relevant again and
again it was worth +0.116 recall. On known-item questions, where the answer is a document you
have not asked about before, it was significantly worse on MRR on all three corpora (-0.035,
-0.065, -0.095). It remembers which documents were useful, which is the wrong memory for a
question you have never asked. So it is a feature the ranker may weigh, not a multiplier applied
to your results.

Exploration is the other trap. Demote a document and it stops being retrieved; stop retrieving it
and it never earns its way back. The multiplier is drawn by Thompson sampling from the posterior,
so a document with thin evidence has a wide posterior and occasionally comes back up anyway.
`explore=0` gives the mean, and is for reproducing an eval rather than for use.

In [ ]:
#| export
@patch
def doc_stats(self:Vault, exclude_ask=None) -> dict:
    """doc_id -> (weighted wins, weighted losses) over the feedback log for this shelf.

    `exclude_ask` drops the rows belonging to given questions, which is what makes an out-of-fold
    prior possible. See `training_data` for why that is not optional."""
    ex = set(exclude_ask or ())
    out = defaultdict(lambda: [0.0, 0.0])
    for r in self._fb()(where=f'store={self.name!r}'):
        if not r['doc_id'] or r['ask_id'] in ex: continue
        w = float(r['weight'] or 1.0)
        out[r['doc_id']][0 if (r['label'] or 0) >= 0.5 else 1] += w
    return {k: tuple(v) for k, v in out.items()}

@patch
def doc_prior(self:Vault,
              gamma:float=0.35,   # how hard the prior is allowed to push; 0 disables it
              explore:float=1.0,  # 1 -> Thompson sample the posterior, 0 -> use its mean
              lo:float=0.6, hi:float=1.6,   # clamp, so no amount of evidence can bury a document
              seed:int=None,
              exclude_ask=None,   # ignore these questions' rows; for an out-of-fold prior
) -> dict:
    "doc_id -> a multiplier on its retrieval score, from the Beta posterior over its feedback."
    rng, out = np.random.default_rng(seed), {}
    for did, (wins, losses) in self.doc_stats(exclude_ask=exclude_ask).items():
        a, b = 1.0 + wins, 1.0 + losses
        p = rng.beta(a, b) if (explore and rng.random() < explore) else a/(a+b)
        out[did] = float(np.clip((p/0.5)**gamma, lo, hi))
    return out

## Is this document noise?

A noisy document is one that comes back for questions it has nothing to do with. Two features
measure that, both computed from vectors already in the file.

`hub` counts how often a chunk appears in other chunks' nearest-neighbour lists. That is the
formal shape of "comes back for everything". 0.984 AUC, the best single feature.

`dup_out` is the fraction of a document's chunks whose nearest neighbour, above a similarity
floor, sits in a different document. Licence blocks, navigation bars and mail footers are
near-duplicates of themselves across a corpus. Real content is not. 0.800.

The obvious feature does not work. "Its chunks are spread across many topics, so it is generic"
scores 0.453 AUC, below chance, and it indicts the wrong documents: a survey paper spans every
cluster and is the last thing you want dropped. That is `spread_doc`, and its default weight is
zero.

The same idea one level down does work. A cookie banner is generic in every chunk it has; a
survey is broad across its chunks and specific inside each one. So measure the entropy of a
single chunk over topic centroids and average that over the document. That is `spread_chunk`,
0.982.

Full table in `evals/RESULTS.md`.

In [ ]:
#| export
#: Every feature is oriented so that larger means noisier, and every one is a document-level
#: aggregate of a chunk-level quantity except where noted.
NOISE_FEATURES = ('hub', 'dup_out', 'off_centre', 'spread_chunk', 'spread_doc', 'low_idf',
                  'redundancy', 'short', 'promiscuity')

#: The default blend, set from the per-feature AUCs in `evals/noise.py` rather than by taste.
#: `spread_doc` is zero because it was measured at 0.54 -- a coin -- and what little it does say
#: points at the surveys. Treat all of these as a starting position: the blend swings between
#: 0.45 and 0.93 AUC across generated corpora, which is the argument for `fit_noise` over any
#: fixed set of numbers, including these.
NOISE_W = dict(hub=1.0, spread_chunk=0.8, dup_out=0.8, off_centre=0.5, low_idf=0.3,
               promiscuity=0.5, redundancy=0.1, spread_doc=0.0, short=0.0)

@patch
def chunk_matrix(self:Vault, limit:int=None, seed:int=0) -> tuple:
    """`(ids, doc_ids, texts, V)`: every chunk's stored vector, L2-normalised, as one array.

    Read off the store rather than re-embedded. The vectors are already there, they are the ones
    retrieval actually uses, and re-embedding a corpus to ask a question about its shape would
    cost more than every other part of this module put together."""
    rows = [r for r in self.store(select='id, doc_id, content, embedding') if r['embedding']]
    if limit and len(rows) > limit:
        idx = np.random.default_rng(seed).choice(len(rows), limit, replace=False)
        rows = [rows[i] for i in sorted(idx)]
    if not rows: return L(), L(), L(), np.zeros((0, 1), np.float32)
    V = np.frombuffer(b''.join(r['embedding'] for r in rows), dtype=DTYPE).reshape(len(rows), -1).astype(np.float32)
    V /= np.linalg.norm(V, axis=1, keepdims=True) + 1e-9
    return (L(r['id'] for r in rows), L(r['doc_id'] for r in rows),
            L(r['content'] or '' for r in rows), V)

### Nearest neighbours

`hub`, `dup_out` and `redundancy` all need every chunk's k nearest neighbours. Exact below
`exact_max` chunks, HNSW above it. Measured here on 256d float16 with cluster structure, 4 cores:

| chunks | blocked numpy | usearch HNSW (build + query) | HNSW recall@10 |
|---|---|---|---|
| 20,000 | 4.8s | 2.4s + 1.1s | 0.68 |
| 50,000 | 33.8s | 10.4s + 4.0s | 0.47 |
| 100,000 | 160.1s | 17.7s + 9.6s | 0.31 |

That recall would be fatal for retrieval. It is survivable here because these features are
aggregates over thousands of chunks, and because HNSW over-returns well-connected points, which
is what a hub is. That is an argument rather than a measurement, so `evals/noise.py` computes
both on a sample and reports the rank correlation.

numkong was the other candidate. It is already in the tree, since usearch pulls it in, and it
does not pay off: 0.32x to 0.79x against numpy on every shape tried, because these are large
matrix multiplies and its edge is per-pair SIMD.

In [ ]:
#| export
def _knn(V:np.ndarray, k:int=10, exact_max:int=30_000, block:int=1024) -> tuple:
    "`(idx, sim)` of each row's `k` nearest others. Exact by blocked matmul, HNSW past `exact_max`."
    n = len(V)
    k = min(k, n - 1)
    if n < 3 or k < 1: return np.zeros((n, 0), np.int32), np.zeros((n, 0), np.float32)
    if n > exact_max:
        from usearch.index import Index
        ix = Index(ndim=V.shape[1], metric='cos', dtype='f32')
        ix.add(np.arange(n), V)
        m = ix.search(V, count=k+1)
        return m.keys[:, 1:].astype(np.int32), (1.0 - m.distances[:, 1:]).astype(np.float32)
    idx, sim = np.empty((n, k), np.int32), np.empty((n, k), np.float32)
    for s in range(0, n, block):
        S = V[s:s+block] @ V.T
        for r in range(S.shape[0]): S[r, s+r] = -2.0        # never your own neighbour
        part = np.argpartition(-S, k, axis=1)[:, :k]
        rows = np.arange(S.shape[0])[:, None]
        ordr = np.argsort(-S[rows, part], axis=1)
        idx[s:s+block] = part[rows, ordr]
        sim[s:s+block] = S[rows, part[rows, ordr]]
    return idx, sim

def _centroids(V:np.ndarray, k:int=64, seed:int=0, iters:int=25) -> np.ndarray:
    """Topic centroids: seeded k-means++ then Lloyd on the unit sphere, L2-normalised.

    Not `usearch.index.kmeans`, which ignores its `seed` -- `spread_doc` is an entropy over these
    assignments and drifted across its whole range between two calls on the same vault. Costs about
    the same, since both are one `V @ C.T` per iteration.

    `k` is capped at a quarter of the chunks. Ask for more centroids than there is structure for and
    every chunk becomes its own cluster, at which point the spread features are zero by
    construction."""
    k = max(2, min(k, len(V) // 4))
    if len(V) <= 8: return V.mean(0, keepdims=True) / (np.linalg.norm(V.mean(0)) + 1e-9)
    rng = np.random.default_rng(seed)
    C = np.empty((k, V.shape[1]), np.float32)
    C[0] = V[rng.integers(len(V))]
    d2 = 1.0 - V @ C[0]                                  # cosine distance, vectors are unit length
    for i in range(1, k):                                # k-means++: far points more likely to be picked
        pr = np.clip(d2, 0, None)**2
        tot = pr.sum()
        C[i] = V[rng.integers(len(V)) if tot <= 0 else int(np.searchsorted(np.cumsum(pr/tot), rng.random()))]
        d2 = np.minimum(d2, 1.0 - V @ C[i])
    prev = None
    for _ in range(iters):
        a = (V @ C.T).argmax(1)
        if prev is not None and np.array_equal(a, prev): break
        prev = a
        for i in range(k):
            m = a == i
            C[i] = V[m].sum(0) if m.any() else V[int(d2.argmax())]
        n = np.linalg.norm(C, axis=1, keepdims=True)
        C = C / np.where(n < 1e-9, 1.0, n)
        d2 = 1.0 - (V @ C.T).max(1)
    n = np.linalg.norm(C, axis=1)
    C = C[n > 1e-6]
    return C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-9)

_TOK = re.compile(r"[A-Za-z0-9_']+")
def _idf(texts) -> dict:
    "Inverse document frequency over chunks: boilerplate is made of words that are everywhere."
    df, n = Counter(), max(len(texts), 1)
    for t in texts: df.update(set(_TOK.findall((t or '').lower())))
    return {w: math.log(n / (1 + c)) for w, c in df.items()}

def _entropy(P:np.ndarray) -> np.ndarray:
    "Normalised Shannon entropy of each row, in [0,1]; 1 means committing to nothing."
    P = np.clip(P, 1e-9, 1.0)
    return (-(P * np.log(P)).sum(1) / math.log(P.shape[1])) if P.shape[1] > 1 else np.zeros(len(P))

In [ ]:
#| export
@patch
def noise_features(self:Vault,
                   k:int=10,           # neighbours per chunk
                   topics:int=64,      # topic centroids for the spread features
                   tau:float=0.9,      # similarity floor for "this is a duplicate"
                   temp:float=8.0,     # softmax temperature for chunk-to-topic spread
                   limit:int=None,     # sample this many chunks instead of using all of them
                   exact_max:int=30_000,
                   seed:int=0,
) -> AttrDict:
    """Per-document noise features, computed from the vectors already in the file.

    Returns `doc_ids`, `X` (documents x `NOISE_FEATURES`) and `names`. No labels are used and no
    model is called, so this is what you have on a vault that has never been given feedback."""
    ids, dids, texts, V = self.chunk_matrix(limit=limit, seed=seed)
    names = list(NOISE_FEATURES)
    if not len(V): return AttrDict(doc_ids=L(), X=np.zeros((0, len(names))), names=names)

    idx, sim = _knn(V, k=k, exact_max=exact_max)
    dida = np.array(dids, dtype=object)
    # a hub is a chunk that turns up in everyone else's neighbour list
    hub = np.bincount(idx.reshape(-1), minlength=len(V)).astype(np.float32) if idx.size else np.zeros(len(V), np.float32)
    if idx.size:
        nb_doc = dida[idx]                                   # whose document each neighbour is in
        same = nb_doc == dida[:, None]
        close = sim >= tau
        dup_out = (close & ~same).any(1).astype(np.float32)  # near-duplicate of another document
        redund = (close & same).any(1).astype(np.float32)    # the document repeating itself
    else: dup_out = redund = np.zeros(len(V), np.float32)

    # distance from, not proximity to. Measured the other way round it scores 0.12 AUC: a survey
    # spanning every topic sits *near* the mean of the corpus and boilerplate sits a long way
    # from it, so proximity to the centroid indicts exactly the documents worth keeping.
    cen = V.mean(0); cen /= np.linalg.norm(cen) + 1e-9
    off_centre = 1.0 - V @ cen
    C_ = _centroids(V, k=topics, seed=seed)
    S = V @ C_.T
    P = np.exp(temp * (S - S.max(1, keepdims=True))); P /= P.sum(1, keepdims=True)
    spread_chunk, assign = _entropy(P), S.argmax(1)

    idf = _idf(texts)
    lo_idf = np.array([-np.mean([idf.get(w, 0.0) for w in _TOK.findall((t or '').lower())] or [0.0])
                       for t in texts], np.float32)
    short = np.array([len(t or '') < 200 for t in texts], np.float32)

    # how many *different* questions this document has been retrieved for, if anything is logged
    seen = defaultdict(set)
    try:
        for r in self._fb()(where=f'store={self.name!r}'):
            if r['doc_id']: seen[r['doc_id']].add(r['q'])
    except Exception: pass
    nq = max(len({q for s in seen.values() for q in s}), 1)

    by = defaultdict(list)
    for i, d in enumerate(dids): by[d].append(i)
    hz = (hub - hub.mean()) / (hub.std() + 1e-9)
    doc_ids, rows = L(), []
    for d, ix in by.items():
        ix = np.array(ix)
        cl = Counter(assign[ix].tolist())
        h = np.array([cl[c] for c in sorted(cl)], np.float32); h /= h.sum()
        rows.append([float(hz[ix].mean()), float(dup_out[ix].mean()), float(off_centre[ix].mean()),
                     float(spread_chunk[ix].mean()), float(_entropy(h[None, :])[0]) if len(h) > 1 else 0.0,
                     float(lo_idf[ix].mean()), float(redund[ix].mean()), float(short[ix].mean()),
                     len(seen.get(d, ())) / nq])
        doc_ids.append(d)
    return AttrDict(doc_ids=doc_ids, X=np.array(rows, np.float32), names=names)

def _robust_z(X:np.ndarray) -> np.ndarray:
    """Per-feature normalisation by rank, centred on zero and spanning roughly [-1.7, 1.7].

    Median/MAD is the obvious choice and it fails here. Boilerplate makes a feature bimodal: with
    40% of documents carrying a glued-on footer the median of `dup_out` sits inside the noisy mode
    and its MAD is near zero, so a perfectly separating feature z-scores to nothing. Ranks are
    invariant to any monotone transform, which is all a blend of unlike features can assume."""
    X = np.asarray(X, np.float64)
    if len(X) < 3: return np.zeros_like(X)
    out = np.empty_like(X)
    for j in range(X.shape[1]):
        col = X[:, j]
        order = np.argsort(col, kind='mergesort')
        r = np.empty(len(col)); r[order] = np.arange(len(col), dtype=float)
        u = np.unique(col)
        if len(u) > 1:                       # average the ranks of ties, so constants stay constant
            for v in u[np.array([(col == v).sum() > 1 for v in u])]:
                m = col == v; r[m] = r[m].mean()
        out[:, j] = 0.0 if len(u) == 1 else (r/(len(col)-1) - 0.5) * 3.46
    return out

@patch
def noise_scores(self:Vault, weights:dict=None, ranker=None, **kw) -> L:
    """Every document, most suspicious first, with the features that put it there.

    `ranker=` takes what `fit_noise` returned and uses it instead of `NOISE_W`. The reported
    per-feature numbers are rank-normalised, so they are comparable down a column and across
    features, and are not the raw values."""
    f = self.noise_features(**kw)
    if not len(f.doc_ids): return L()
    Z = _robust_z(f.X)
    if ranker is not None: s = ranker.score(Z)
    else:
        w = np.array([(weights or NOISE_W).get(n, 0.0) for n in f.names], np.float32)
        s = Z @ w / (np.abs(w).sum() or 1.0)
    titles = {r['id']: r['title'] for r in self.t.docs(select='id, title')}
    out = L(AttrDict(doc_id=d, title=titles.get(d, d), score=float(sc),
                     **{n: float(v) for n, v in zip(f.names, x)})
            for d, sc, x in zip(f.doc_ids, s, Z))
    return out.sorted(key=lambda r: -r.score)

@patch
def suggest_noisy(self:Vault, k:int=20, min_score:float=1.0, **kw) -> L:
    """The `k` documents most worth looking at, excluding ones already judged.

    A suggestion and nothing else. Confirming one is `v.mark_noisy(doc_id)`, which is a person
    deciding: the score never removes anything from your results on its own, because the cost of
    silently hiding the one document that mattered is not symmetric with the cost of a bad hit."""
    judged = {r['doc_id'] for r in self._marks()(where=f'store={self.name!r}')}
    return self.noise_scores(**kw).filter(lambda r: r.score >= min_score and r.doc_id not in judged)[:k]

## The ranker

Pairwise and linear, fitted on the log. The label is an ordering inside one question: this
section was cited, that one was not. Relevance is not comparable across questions, so there is no
score to regress on.

For a linear scorer that is convenient. Ranking i above j means `w.(xi - xj) > 0`, which is
logistic regression on feature differences with no intercept, since the intercept cancels. Fitted
by Newton's method: twenty-odd features means a 20x20 Hessian and under ten iterations.

Read the evals before switching it on. None of the three models beat plain RRF reproducibly on
novel questions. Mean nDCG@10 against baseline, three corpora, known-item queries,
document-disjoint splits: -0.127 for the linear model applied on its own, -0.085 for a random
forest, -0.005 for gradient boosting. Gradient boosting won MRR outright on one corpus (+0.083,
p=0.05) and lost it outright on another (-0.109, p=0.01), which is what variance looks like.

The only configuration that never lost anywhere was the linear model fused with the incoming
order rather than substituted for it. It never won either. `fit_ranker` and `use_ranker` are two
calls because of that.

In [ ]:
#| export
def _sig(z): return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

class Ranker:
    """Pairwise linear learning-to-rank over the feedback log.

    `fit` takes a design matrix, a group id per row (the question the row was retrieved for) and a
    label per row, and learns weights such that higher-labelled rows in the same group score
    higher. `score` applies them. Both directions of every pair are implied by symmetry, so only
    one is materialised."""
    def __init__(self, names, w=None, mu=None, sd=None, meta=None):
        self.names, self.w = list(names), None if w is None else np.asarray(w, np.float64)
        self.mu = None if mu is None else np.asarray(mu, np.float64)
        self.sd = None if sd is None else np.asarray(sd, np.float64)
        self.meta = dict(meta or {})

    def __repr__(self):
        if self.w is None: return f'Ranker({len(self.names)} features, unfitted)'
        top = sorted(zip(self.names, self.w), key=lambda t: -abs(t[1]))[:5]
        return 'Ranker(' + ', '.join(f'{n}={v:+.2f}' for n, v in top) + ', ...)'

    def _pairs(self, X, groups, y, wt, max_pairs, seed):
        by = defaultdict(list)
        for i, g in enumerate(groups): by[g].append(i)
        pi, pj, pw = [], [], []
        for ix in by.values():
            for a in ix:
                for b in ix:
                    if y[a] > y[b]: pi.append(a); pj.append(b); pw.append(min(wt[a], wt[b]))
        if not pi: return None
        pi, pj, pw = np.array(pi), np.array(pj), np.array(pw, np.float64)
        if len(pi) > max_pairs:
            s = np.random.default_rng(seed).choice(len(pi), max_pairs, replace=False)
            pi, pj, pw = pi[s], pj[s], pw[s]
        return pi, pj, pw

    def fit(self, X, groups, y, weights=None, l2:float=1.0, iters:int=25, tol:float=1e-7,
            max_pairs:int=200_000, seed:int=0):
        "Fit by IRLS on the pairwise logistic loss. Returns self, or self unfitted if no pair exists."
        X = np.asarray(X, np.float64); y = np.asarray(y, np.float64)
        wt = np.ones(len(X)) if weights is None else np.asarray(weights, np.float64)
        self.mu, self.sd = X.mean(0), X.std(0) + 1e-9
        Z = (X - self.mu) / self.sd
        p = self._pairs(Z, groups, y, wt, max_pairs, seed)
        if p is None:
            warnings.warn('no ordered pair in the feedback: nothing to fit'); return self
        pi, pj, pw = p
        D, w = Z[pi] - Z[pj], np.zeros(Z.shape[1])
        n, I = len(D), np.eye(Z.shape[1])
        for _ in range(iters):
            mu = _sig(D @ w)
            g = D.T @ (pw * (mu - 1.0)) / n + l2 * w          # every pair's target is 1
            H = (D * (pw * mu * (1 - mu))[:, None]).T @ D / n + l2 * I
            step = np.linalg.solve(H, g)
            w -= step
            if np.linalg.norm(step) < tol: break
        self.w = w
        self.meta.update(n_pairs=int(n), n_rows=int(len(X)), n_groups=len(set(groups)), l2=l2)
        return self

    def score(self, X):
        "Higher is better. Unfitted, everything ties: so a caller can always ask."
        X = np.asarray(X, np.float64)
        if self.w is None: return np.zeros(len(X))
        return ((X - self.mu) / self.sd) @ self.w

    __call__ = score
    def weights(self) -> L:
        "The fitted weights, largest first: the thing you read before trusting any of this."
        if self.w is None: return L()
        return L(sorted(zip(self.names, self.w.tolist()), key=lambda t: -abs(t[1]))).map(
            lambda t: AttrDict(feature=t[0], weight=round(t[1], 4)))

    def to_dict(self): return dict(names=self.names, w=None if self.w is None else self.w.tolist(),
                                   mu=None if self.mu is None else self.mu.tolist(),
                                   sd=None if self.sd is None else self.sd.tolist(), meta=self.meta)
    @classmethod
    def from_dict(cls, d): return cls(d['names'], d.get('w'), d.get('mu'), d.get('sd'), d.get('meta'))

### The features

Three groups, and the split matters when you read the weights. Retrieval (`score`, `rank`, `gap`)
says where RRF put the section, so a model leaning on those has learned nothing. Match
(`overlap`, `idf_overlap`, `cos`, `len`) is about this question and this section. Document
(`prior`, the nine noise features, kind, age) is about the document whatever the question.

The document group is where leakage lives. `prior` is computed out of fold in `training_data`,
and `evals/run.py` reports a document-disjoint split, because a model that has memorised which
documents are good looks fine on a query split.

In [ ]:
#| export
def _epoch(v) -> float:
    "`added_at` is an epoch on some rows and a SQL timestamp string on others. Take either."
    if v is None: return time.time()
    try: return float(v)
    except (TypeError, ValueError): pass
    for fmt in ('%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M:%S.%f', '%Y-%m-%dT%H:%M:%S'):
        try: return time.mktime(time.strptime(str(v)[:26], fmt))
        except ValueError: continue
    return time.time()

PAIR_FEATURES = ('score', 'rank', 'gap', 'overlap', 'idf_overlap', 'cos', 'len',
                 'prior', 'is_code', 'is_web', 'is_note', 'age') + NOISE_FEATURES

@patch
def _doc_side(self:Vault, refresh:bool=False) -> tuple:
    "Cached per-document features: the noise block, the Beta prior, kind and age."
    if refresh or getattr(self, '_ds', None) is None:
        f = self.noise_features()
        nz = dict(zip(f.doc_ids, _robust_z(f.X))) if len(f.doc_ids) else {}
        meta = {r['id']: (r['kind'] or '', _epoch(r['added_at'])) for r in self.t.docs(select='id, kind, added_at')}
        self._ds = (nz, meta)
    return self._ds

@patch
def pair_X(self:Vault, q:str, hits, prior:dict=None) -> np.ndarray:
    "The design matrix for one question's hits, in `PAIR_FEATURES` order."
    nz, meta = self._doc_side()
    prior = self.doc_prior() if prior is None else prior
    idf = getattr(self, '_idf_cache', None)
    if idf is None: idf = self._idf_cache = _idf(self.chunk_matrix(limit=20_000)[2])
    qt = set(_TOK.findall((q or '').lower()))
    qw = sum(idf.get(w, 0.0) for w in qt) or 1.0
    qv = np.frombuffer(self.qemb(q), dtype=DTYPE).astype(np.float32) if q else None
    if qv is not None: qv /= np.linalg.norm(qv) + 1e-9
    top = max((float(getattr(h, 'score', 0.0) or 0.0) for h in hits), default=0.0) or 1.0
    now, rows = time.time(), []
    for i, h in enumerate(hits, 1):
        txt = str(getattr(h, 'text', '') or '')
        ht = set(_TOK.findall(txt.lower()))
        sc = float(getattr(h, 'score', 0.0) or 0.0)
        did = getattr(h, 'doc_id', None)
        kind, added = meta.get(did, ('', now))
        cos = 0.0
        if qv is not None and getattr(h, 'node_id', None):
            vs = [np.frombuffer(r['embedding'], dtype=DTYPE).astype(np.float32)
                  for r in self.store(where=f"node_id={h.node_id!r}", select='embedding') if r['embedding']]
            if vs:
                m = np.mean(vs, 0); cos = float(qv @ (m / (np.linalg.norm(m) + 1e-9)))
        rows.append([sc, 1.0/i, sc/top,
                     len(qt & ht)/max(len(qt), 1),
                     sum(idf.get(w, 0.0) for w in qt & ht)/qw,
                     cos, math.log1p(len(txt)),
                     float(prior.get(did, 1.0)),
                     float(kind == 'code'), float(kind == 'web'), float(kind == 'note'),
                     math.log1p(max(now - added, 0)/86400.0)]
                    + list(nz.get(did, np.zeros(len(NOISE_FEATURES)))))
    return np.array(rows, np.float64) if rows else np.zeros((0, len(PAIR_FEATURES)))

## Fitting, storing, switching on

Three calls rather than one. `fit_ranker` produces a model and its weights. `use_ranker` puts it
in the path of your questions. Whether it belongs there is a question for `evals/`.

In [ ]:
#| export
@patch
def _rankers(self:Vault):
    "Where a fitted ranker and the logging switch live, per shelf."
    t = self.db.t.rankers
    t.create(store=str, model=str, at=float, enabled=int, logging=int, note=str,
             pk='store', if_not_exists=True)
    return t

@patch
def learn(self:Vault, on:bool=True) -> dict:
    "Log every `ask` as feedback from now on. Off by default: nothing is recorded until you say so."
    t = self._rankers()
    row = dict(first(t(where=f'store={self.name!r}')) or dict(store=self.name, model='', at=time.time(),
                                                              enabled=0, note=''))
    row.update(logging=int(bool(on)), store=self.name)
    t.insert(row, replace=True); self._learning = bool(on)
    return row

@patch
def training_data(self:Vault, min_group:int=2) -> AttrDict:
    """The feedback log as `(X, groups, y, weights)`, one group per question.

    Questions with a single distinct label are dropped: a pairwise model learns from disagreement
    inside a group, and a group where everything was cited yields no pair.

    `prior` is computed out of fold, from a posterior excluding that question's own rows. In fold
    it is target leakage: the feature summarises the label, the fit puts its largest weight on it,
    and the result scored recall 0.47 against a baseline of 0.80 on a document-disjoint split."""
    by = defaultdict(list)
    for r in self._fb()(where=f'store={self.name!r}', order_by='at'): by[r['ask_id']].append(r)
    Xs, gs, ys, ws = [], [], [], []
    nodes = {}
    for aid, rows in by.items():
        labs = {float(r['label'] or 0) for r in rows}
        if len(rows) < min_group or len(labs) < 2: continue
        hits = L()
        for r in rows:
            if r['node_id'] not in nodes:
                nodes[r['node_id']] = (self.read(r['node_id'], max_chars=4000) or {}).get('text', '')
            hits.append(AttrDict(node_id=r['node_id'], doc_id=r['doc_id'], text=nodes[r['node_id']],
                                 score=float(r['score'] or 0.0)))
        X = self.pair_X(rows[0]['q'], hits, prior=self.doc_prior(explore=0.0, exclude_ask={aid}))
        if not len(X): continue
        Xs.append(X); gs += [aid]*len(X)
        ys += [float(r['label'] or 0) for r in rows]; ws += [float(r['weight'] or 1.0) for r in rows]
    if not Xs: return AttrDict(X=np.zeros((0, len(PAIR_FEATURES))), groups=[], y=[], weights=[],
                               names=list(PAIR_FEATURES))
    return AttrDict(X=np.vstack(Xs), groups=gs, y=np.array(ys), weights=np.array(ws),
                    names=list(PAIR_FEATURES))

@patch
def fit_ranker(self:Vault, l2:float=1.0, save:bool=False, **kw) -> Ranker:
    "Fit a `Ranker` on everything logged so far. `save=True` stores it; it is still not switched on."
    d = self.training_data()
    r = Ranker(d.names).fit(d.X, d.groups, d.y, weights=d.weights, l2=l2, **kw)
    if save and r.w is not None:
        t = self._rankers()
        row = dict(first(t(where=f'store={self.name!r}')) or dict(store=self.name, enabled=0, logging=0, note=''))
        row.update(store=self.name, model=json.dumps(r.to_dict()), at=time.time())
        t.insert(row, replace=True)
        self._rk_cache = None
    return r

@patch
def fit_noise(self:Vault, labels:dict=None, l2:float=1.0, **kw) -> Ranker:
    """Learn the blend from the documents you have marked, instead of guessing nine weights.

    Same pairwise machinery as the ranker: one group holding every document, marked ones labelled
    1. "Rank every noisy document above every clean one" is the AUC, so the pairwise loss optimises
    what `evals/noise.py` reports. Held out, this beat the fixed weights on all four corpora tried
    (0.996 against 0.979) from six marks. The fixed weights are the cold start."""
    labels = labels or {r['doc_id']: bool(r['noisy']) for r in self._marks()(where=f'store={self.name!r}')
                        if r['noisy'] is not None}
    if not labels: raise ValueError('nothing marked: mark_noisy a few documents first')
    f = self.noise_features(**kw)
    keep = [i for i, d in enumerate(f.doc_ids) if d in labels] or list(range(len(f.doc_ids)))
    y = np.array([float(labels.get(f.doc_ids[i], 0.0)) for i in range(len(f.doc_ids))])
    if len(set(y[keep].tolist())) < 2:
        # only positives marked: the unmarked documents are the negatives, which is what a person
        # marking noise actually means, and is also the only way to get a pair out of this
        keep = list(range(len(f.doc_ids)))
    X = _robust_z(f.X)[keep]
    return Ranker(f.names).fit(X, ['all']*len(keep), y[keep], l2=l2)

@patch
def use_ranker(self:Vault, on:bool=True) -> dict:
    "Put the stored ranker in the path of `context`, and so of `ask`. Separate from fitting it."
    t = self._rankers()
    row = first(t(where=f'store={self.name!r}'))
    if not row or not row['model']: raise ValueError('no ranker stored; fit_ranker(save=True) first')
    row = dict(row, enabled=int(bool(on)))
    t.insert(row, replace=True); self._rk_cache = None
    return dict(store=self.name, enabled=bool(on), fitted_at=row['at'])

@patch
def _rk(self:Vault):
    "The enabled ranker for this shelf, or None. Read once per Vault, not once per question."
    if getattr(self, '_rk_cache', None) is None:
        try:
            r = first(self._rankers()(where=f'store={self.name!r} AND enabled=1'))
            self._rk_cache = Ranker.from_dict(json.loads(r['model'])) if (r and r['model']) else False
        except Exception: self._rk_cache = False
    return self._rk_cache or None

@patch
def retune(self:Vault, q:str, hits, ranker=None, alpha:float=0.25, rrf_k:int=60) -> L:
    """Reorder one question's hits, fused with the order they arrived in by reciprocal rank.

    Fused rather than substituted. A ranker that sorts by its own score discards the baseline, so a
    mediocre model does not degrade the ranking, it destroys it: over three corpora, substituting
    cost a mean nDCG@10 of -0.127 and lost significantly on two, where the same model at
    `alpha=0.25` came out at -0.001 and was never significant either way.

    RRF is what the two retrieval legs are already fused with a layer down, so this is a third leg
    with a weight on it. `alpha=0` leaves the baseline untouched, large `alpha` is the ranker alone."""
    rk = ranker or self._rk()
    if rk is None or not len(hits): return L(hits)
    s = rk.score(self.pair_X(q, hits))
    order = sorted(range(len(hits)), key=lambda i: -s[i])
    lrank = {i: r for r, i in enumerate(order)}
    for i, h in enumerate(hits):
        h.tuned = float(s[i])
        h.fused = 1.0/(rrf_k + i) + alpha/(rrf_k + lrank[i])
    return L(sorted(hits, key=lambda h: -h.fused))

@patch
def _post(self:Vault, q:str, ctx):
    "The hook `context` calls: a no-op until a ranker has been fitted *and* switched on."
    if self._rk() is None: return ctx
    try: ctx.results, ctx.tuned = self.retune(q, ctx.results), True
    except Exception as e: warnings.warn(f'ranker not applied ({type(e).__name__}: {e})')
    return ctx

## A worked check

Small, synthetic, offline. Enough to show the shapes line up. It does not show that any of this
improves retrieval; that is `evals/`, which needs a real corpus.

In [ ]:
#| hide
import tempfile
from pathlib import Path
v = Vault(Path(tempfile.mkdtemp())/'q.db', offline=True, dims=64)
for i in range(6):
    v.add(f'Document {i} about hybrid retrieval, reciprocal rank fusion and chunk size {i}. '
          f'Section two of document {i} discusses evaluation and mean reciprocal rank.',
          title=f'Real {i}', source=f'real{i}')
for i in range(4):
    v.add('Cookie policy. All rights reserved. Contact us. Privacy. Terms of service. '
          'Copyright notice. All rights reserved. Cookie policy.', title=f'Boiler {i}', source=f'b{i}')
f = v.noise_features()
assert f.X.shape == (10, len(NOISE_FEATURES)), f.X.shape
sc = v.noise_scores()
print('most suspicious:', [(r.title, round(r.score, 2)) for r in sc[:3]])
assert sum('Boiler' in r.title for r in sc[:4]) >= 3, 'boilerplate should surface'


In [ ]:
#| hide
# a fabricated feedback log, only to show the fit runs and the weights come out readable
import uuid as _uuid
for qi, q in enumerate(['reciprocal rank fusion', 'chunk size', 'evaluation mrr']):
    hits = v.sections(q, limit=6)
    aid = _uuid.uuid4().hex[:12]
    for i, s in enumerate(hits, 1):
        good = 'Boiler' not in (s['breadcrumb'] or '')
        v.rate(q, node_id=s['node_id'], signal='cited' if good else 'shown', rank=i,
               score=float(s['score']), ask_id=aid)
d = v.training_data()
print('training rows', d.X.shape, 'groups', len(set(d.groups)))
r = v.fit_ranker()
print(r)
assert r.w is not None and len(r.w) == len(PAIR_FEATURES)
print(r.weights()[:4])